In [1]:
%pip install pandas pyarrow duckdb ijson

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from collections import Counter
import json
import csv
import os
import re
import warnings

import pandas as pd
import pyarrow
import duckdb
import ijson

warnings.filterwarnings("default")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)
print("DuckDB:", duckdb.__version__)
print("ijson:", ijson.__version__ if hasattr(ijson, "__version__") else "installed")

Pandas: 2.3.3
PyArrow: 21.0.0
DuckDB: 1.5.6
ijson: 3.6.0


In [3]:
cwd = Path.cwd().resolve()

if (cwd / "raw").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "raw").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Не найдена папка raw/. "
        "Проверь расположение ноутбука относительно корня проекта."
    )

RAW_DIR = PROJECT_ROOT / "raw"
PROCESSED_DIR = PROJECT_ROOT / "processed"
REPORTS_DIR = PROJECT_ROOT / "docs" / "data_quality"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", PROJECT_ROOT)
print("RAW:", RAW_DIR)
print("Processed:", PROCESSED_DIR)
print("Отчёты:", REPORTS_DIR)

Корень проекта: C:\Users\245108\Desktop\gaming-data-analysis
RAW: C:\Users\245108\Desktop\gaming-data-analysis\raw
Processed: C:\Users\245108\Desktop\gaming-data-analysis\processed
Отчёты: C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality


In [4]:
def get_file_inventory(root: Path) -> pd.DataFrame:
    records = []

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        try:
            size_bytes = path.stat().st_size

            records.append({
                "file": path.name,
                "relative_path": str(path.relative_to(root)),
                "extension": path.suffix.lower() or "[без расширения]",
                "size_bytes": size_bytes,
                "size_mb": round(size_bytes / 1024**2, 2),
                "size_gb": round(size_bytes / 1024**3, 3),
            })

        except OSError as exc:
            print(f"Не удалось прочитать метаданные {path}: {exc}")

    columns = [
        "file", "relative_path", "extension",
        "size_bytes", "size_mb", "size_gb"
    ]

    return pd.DataFrame(records, columns=columns)


inventory = get_file_inventory(RAW_DIR)

display(
    inventory.sort_values("size_bytes", ascending=False)
    .reset_index(drop=True)
)

print(f"Всего файлов: {len(inventory):,}")
print(f"Общий объём: {inventory['size_gb'].sum():.2f} ГБ")

print("\nКоличество файлов по расширениям:")
display(
    inventory.groupby("extension")
    .agg(
        files=("file", "count"),
        total_mb=("size_mb", "sum"),
    )
    .sort_values("total_mb", ascending=False)
)

,file,relative_path,extension,size_bytes,size_mb,size_gb
0,videogames_data.jsonl,rawg\jsonl\videogames_data.jsonl,.jsonl,4300705739,4101.47,4.005
1,dataset.csv,steam\reviews\dataset.csv,.csv,2160871419,2060.77,2.012
2,recommendations.csv,steam\recommendations\recommendations.csv,.csv,2023223067,1929.50,1.884
3,videogames_data.csv,rawg\videogames_data.csv,.csv,2000469667,1907.80,1.863
4,videogames_data.parquet,rawg\videogames_data.parquet,.parquet,1011662649,964.80,0.942
5,games.json,steam\fronkongames\games.json,.json,970222554,925.28,0.904
6,reviews.csv,steam\games_2025\steam_dataset_2025_csv\reviews.csv,.csv,669812331,638.78,0.624
7,games_march2025_full.csv,steam\games_artemiloff\games_march2025_full.csv,.csv,471492150,449.65,0.439
8,games_march2025_cleaned.csv,steam\games_artemiloff\games_march2025_cleaned.csv,.csv,468641107,446.93,0.436
9,games_may2024_full.csv,steam\games_artemiloff\games_may2024_full.csv,.csv,424919226,405.23,0.396


Всего файлов: 42
Общий объём: 14.76 ГБ

Количество файлов по расширениям:


,files,total_mb
extension,,
.csv,36,9101.71
.jsonl,1,4101.47
.parquet,1,964.80
.json,3,942.02
.py,1,0.01


In [5]:
inventory.to_csv(
    REPORTS_DIR / "raw_file_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Инвентаризация сохранена.")

Инвентаризация сохранена.


In [6]:
EXPECTED_SOURCES = {
    "rawg_games": "rawg",
    "steam_games_fronkongames": "steam/fronkongames",
    "steam_reviews": "steam/reviews",
    "steam_recommendations": "steam/recommendations",
    "steam_games_2025": "steam/games_2025",
    "steam_games_2021_2025": "steam/games_2021_2025",
    "steam_games_artemiloff": "steam/games_artemiloff",
    "steam_monthly_players": "steam/monthly_players",
    "metacritic_vs_steam": "steam/metacritic_vs_steam",
    "ultimate_games": "games/ultimate_games",
    "twitch_streamers": "twitch/streamers",
    "twitch_evolution": "twitch/evolution",
    "epic_games": "epic",
}

source_check = []

for source, relative_dir in EXPECTED_SOURCES.items():
    source_path = RAW_DIR / relative_dir

    files = [
        path for path in source_path.rglob("*")
        if path.is_file()
    ] if source_path.exists() else []

    source_check.append({
        "source": source,
        "expected_directory": relative_dir,
        "directory_exists": source_path.exists(),
        "file_count": len(files),
        "total_mb": round(
            sum(path.stat().st_size for path in files) / 1024**2, 2
        ),
    })

source_check = pd.DataFrame(source_check)

display(source_check)

print(
    "Источники без найденных файлов:",
    source_check.loc[
        source_check["file_count"] == 0, "source"
    ].tolist()
)

,source,expected_directory,directory_exists,file_count,total_mb
0,rawg_games,rawg,True,3,6974.07
1,steam_games_fronkongames,steam/fronkongames,True,2,1307.83
2,steam_reviews,steam/reviews,True,1,2060.77
3,steam_recommendations,steam/recommendations,True,4,2134.99
4,steam_games_2025,steam/games_2025,True,13,841.52
5,steam_games_2021_2025,steam/games_2021_2025,True,2,11.12
6,steam_games_artemiloff,steam/games_artemiloff,True,4,1704.57
7,steam_monthly_players,steam/monthly_players,True,1,36.13
8,metacritic_vs_steam,steam/metacritic_vs_steam,True,4,9.51
9,ultimate_games,games/ultimate_games,True,1,27.54


Источники без найденных файлов: []


## Инвентаризация по csv

In [7]:
def inspect_csv_header(path: Path) -> dict:
    result = {
        "file": path.name,
        "relative_path": str(path.relative_to(RAW_DIR)),
        "columns_count": None,
        "columns": None,
        "encoding": None,
        "delimiter": None,
        "error": None,
    }

    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            with path.open(
                "r", encoding=encoding, newline=""
            ) as file:
                sample = file.read(65536)

                if not sample.strip():
                    result["error"] = "Пустой файл"
                    return result

                try:
                    dialect = csv.Sniffer().sniff(
                        sample, delimiters=",;\t|"
                    )
                    delimiter = dialect.delimiter
                except csv.Error:
                    delimiter = ","

                reader = csv.reader(
                    [sample.splitlines()[0]],
                    delimiter=delimiter,
                )
                columns = next(reader)

                result.update({
                    "columns_count": len(columns),
                    "columns": columns,
                    "encoding": encoding,
                    "delimiter": repr(delimiter),
                })

                return result

        except UnicodeDecodeError:
            continue
        except Exception as exc:
            result["error"] = str(exc)
            return result

    result["error"] = "Не удалось определить кодировку"
    return result


csv_files = [
    RAW_DIR / relative
    for relative in inventory.loc[
        inventory["extension"] == ".csv", "relative_path"
    ]
]

csv_schema_records = [
    inspect_csv_header(path)
    for path in csv_files
]

csv_schemas = pd.DataFrame(csv_schema_records)

display(
    csv_schemas[
        ["relative_path", "columns_count", "columns",
         "encoding", "delimiter", "error"]
    ]
)

,relative_path,columns_count,columns,encoding,delimiter,error
0,epic\epic games store-video games.csv,17,"[game_link, name, price of game, developer of game, genres of games, features of game, publisher...",utf-8-sig,"','",None
1,epic\epic-games-apps.csv,16,"[links, name, type, genres, features, price, developer, publisher, release_date, platform, os re...",utf-8-sig,"','",None
2,epic\epic-games-demo.csv,17,"[links, name, type, genres, features, Developer, publisher, release date, initial release in win...",utf-8-sig,"','",None
3,epic\epic_games_editor.csv,20,"[links, name, type, genres, features, Developer, publisher, release date, platform, os recommend...",utf-8-sig,"','",None
4,rawg\videogames_data.csv,54,"[id, slug, name, released, background_image, rating, rating_top, ratings_count, reviews_text_cou...",utf-8-sig,"','",None
5,twitch\evolution\Twitch_game_data.csv,12,"[Rank, Game, Month, Year, Hours_watched, Hours_streamed, Peak_viewers, Peak_channels, Streamers,...",utf-8-sig,"','",None
6,twitch\evolution\Twitch_global_data.csv,9,"[year, Month, Hours_watched, Avg_viewers, Peak_viewers, Streams, Avg_channels, Games_streamed, V...",utf-8-sig,"','",None
7,twitch\streamers\datasetV2.csv,17,"[RANK, NAME, LANGUAGE, TYPE, MOST_STREAMED_GAME, 2ND_MOST_STREAMED_GAME, AVERAGE_STREAM_DURATION...",utf-8-sig,"','",None
8,steam\fronkongames\games.csv,39,"[AppID, Name, Release date, Estimated owners, Peak CCU, Required age, Price, DiscountDLC count, ...",utf-8-sig,"','",None
9,steam\games_2021_2025\a_steam_data_2021_2025.csv,10,"[appid, name, release_year, release_date, genres, categories, price, recommendations, developer,...",utf-8-sig,"','",None


## Инвентаризация по JSON

In [8]:
def find_file_by_name(root: Path, filename: str):
    matches = list(root.rglob(filename))

    if not matches:
        print(f"Файл не найден: {filename}")
        return None

    if len(matches) > 1:
        print(f"Найдено несколько файлов {filename}:")
        for path in matches:
            print(" ", path.relative_to(root))

    return matches[0]


games_json_path = find_file_by_name(RAW_DIR, "games.json")
rawg_jsonl_path = find_file_by_name(RAW_DIR, "videogames_data.jsonl")


def inspect_file_prefix(path: Path, n_bytes=300):
    if path is None:
        return

    with path.open("rb") as file:
        prefix = file.read(n_bytes)

    print(f"\nФайл: {path.relative_to(RAW_DIR)}")
    print("Размер, МБ:", round(path.stat().st_size / 1024**2, 2))
    print("Начало файла:")
    print(prefix.decode("utf-8", errors="replace"))


inspect_file_prefix(games_json_path)
inspect_file_prefix(rawg_jsonl_path)


Файл: steam\fronkongames\games.json
Размер, МБ: 925.28
Начало файла:
{
    "2539430": {
        "name": "Black Dragon Mage Playtest",
        "release_date": "Aug 1, 2023",
        "required_age": 0,
        "price": 0.0,
        "dlc_count": 0,
        "detailed_description": "",
        "about_the_game": "",
        "short_description": "",
        "reviews": "",
 

Файл: rawg\jsonl\videogames_data.jsonl
Размер, МБ: 4101.47
Начало файла:
{"id":3498,"slug":"grand-theft-auto-v","name":"Grand Theft Auto V","released":"2013-09-17","background_image":"https://media.rawg.io/media/games/20a/20aa03a10cda45239fe22d035c0ebe64.jpg","rating":4.47,"rating_top":5,"ratings_count":7409,"reviews_text_count":135,"added":22635,"metacritic":92,"playtim


In [9]:
if games_json_path is not None:
    with games_json_path.open("rb") as file:
        app_id, game = next(ijson.kvitems(file, ""))

    print("Пример AppID:", app_id)
    print("Тип записи:", type(game).__name__)
    print("Число полей:", len(game))
    print("Поля:", list(game.keys()))

Пример AppID: 2539430
Тип записи: dict
Число полей: 42
Поля: ['name', 'release_date', 'required_age', 'price', 'dlc_count', 'detailed_description', 'about_the_game', 'short_description', 'reviews', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'achievements', 'recommendations', 'notes', 'supported_languages', 'full_audio_languages', 'packages', 'developers', 'publishers', 'categories', 'genres', 'screenshots', 'movies', 'user_score', 'score_rank', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu', 'tags']


In [10]:
if rawg_jsonl_path is not None:
    with rawg_jsonl_path.open(
        "r", encoding="utf-8"
    ) as file:
        for i, line in enumerate(file):
            if not line.strip():
                continue

            try:
                record = json.loads(line)
                print(f"\nЗапись {i + 1}")
                print("Тип:", type(record).__name__)

                if isinstance(record, dict):
                    print("Поля:", list(record.keys())[:50])
                else:
                    print("Пример:", str(record)[:300])

            except json.JSONDecodeError as exc:
                print("Ошибка JSON в строке", i + 1, exc)

            if i >= 2:
                break


Запись 1
Тип: dict
Поля: ['id', 'slug', 'name', 'released', 'background_image', 'rating', 'rating_top', 'ratings_count', 'reviews_text_count', 'added', 'metacritic', 'playtime', 'suggestions_count', 'updated', 'reviews_count', 'saturated_color', 'dominant_color', 'platforms', 'stores', 'developers', 'genres', 'tags', 'publishers', 'esrb_rating', 'added_by_status', 'metacritic_url', 'ratings', 'clip', 'name_original', 'reddit_url', 'movies_count', 'reactions', 'background_image_additional', 'website', 'reddit_count', 'achievements_count', 'youtube_count', 'short_screenshots', 'creators_count', 'twitch_count', 'alternative_names', 'parent_achievements_count', 'additions_count', 'metacritic_platforms', 'screenshots_count', 'parent_platforms', 'game_series_count', 'description', 'description_raw']

Запись 2
Тип: dict
Поля: ['id', 'slug', 'name', 'released', 'background_image', 'rating', 'rating_top', 'ratings_count', 'reviews_text_count', 'added', 'metacritic', 'playtime', 'suggestions_co

In [11]:
csv_schemas.to_csv(
    REPORTS_DIR / "csv_schemas.csv",
    index=False,
    encoding="utf-8-sig",
)

source_check.to_csv(
    REPORTS_DIR / "source_check.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Сохранены:")
print("-", REPORTS_DIR / "raw_file_inventory.csv")
print("-", REPORTS_DIR / "csv_schemas.csv")
print("-", REPORTS_DIR / "source_check.csv")

Сохранены:
- C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\raw_file_inventory.csv
- C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\csv_schemas.csv
- C:\Users\245108\Desktop\gaming-data-analysis\docs\data_quality\source_check.csv


## Дмагностика качества строк

In [13]:
import csv
from pathlib import Path
import pandas as pd


def check_csv_structure(path: Path, max_records: int = 1000) -> dict:
    result = {
        "file": str(path.relative_to(RAW_DIR)),
        "header_columns": None,
        "records_checked": 0,
        "bad_records": 0,
        "examples": [],
        "error": None,
    }

    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin1"):
        try:
            with path.open(
                "r", encoding=encoding, newline=""
            ) as file:
                reader = csv.reader(file)

                header = next(reader, None)

                if header is None:
                    result["error"] = "Пустой файл"
                    return result

                result["header_columns"] = len(header)

                for row_number, row in enumerate(
                    reader, start=2
                ):
                    result["records_checked"] += 1

                    if len(row) != len(header):
                        result["bad_records"] += 1

                        if len(result["examples"]) < 5:
                            result["examples"].append({
                                "row": row_number,
                                "expected_fields": len(header),
                                "actual_fields": len(row),
                            })

                    if result["records_checked"] >= max_records:
                        break

            return result

        except UnicodeDecodeError:
            continue
        except Exception as exc:
            result["error"] = str(exc)
            return result

    result["error"] = "Не удалось определить кодировку"
    return result


# Сначала проверяем файлы, которые важнее всего для анализа.
priority_files = [
    "steam/fronkongames/games.csv",
    "steam/artemiloff/games_march2025_full.csv",
    "steam/artemiloff/games_march2025_cleaned.csv",
    "steam/artemiloff/games_may2024_full.csv",
    "steam/artemiloff/games_may2024_cleaned.csv",
    "epic/epic games store-video games.csv",
    "epic/epic-games-apps.csv",
    "epic/epic-games-demo.csv",
    "epic/epic_games_editor.csv",
]

structure_results = []

for relative_path in priority_files:
    path = RAW_DIR / relative_path

    if path.exists():
        structure_results.append(
            check_csv_structure(path)
        )
    else:
        print(f"Файл не найден: {relative_path}")

structure_df = pd.DataFrame(structure_results)

display(structure_df)

Файл не найден: steam/artemiloff/games_march2025_full.csv
Файл не найден: steam/artemiloff/games_march2025_cleaned.csv
Файл не найден: steam/artemiloff/games_may2024_full.csv
Файл не найден: steam/artemiloff/games_may2024_cleaned.csv


,file,header_columns,records_checked,bad_records,examples,error
0,steam\fronkongames\games.csv,39,1000,1000,"[{'row': 2, 'expected_fields': 39, 'actual_fields': 40}, {'row': 3, 'expected_fields': 39, 'actu...",None
1,epic\epic games store-video games.csv,17,899,0,[],None
2,epic\epic-games-apps.csv,16,11,0,[],None
3,epic\epic-games-demo.csv,17,25,0,[],None
4,epic\epic_games_editor.csv,20,20,0,[],None


In [14]:
import pandas as pd

epic_files = [
    "epic games store-video games.csv",
    "epic-games-apps.csv",
    "epic-games-demo.csv",
    "epic_games_editor.csv",
]

epic_frames = {}
epic_summary = []

for filename in epic_files:
    path = RAW_DIR / "epic" / filename

    if not path.exists():
        continue

    df = pd.read_csv(
        path,
        encoding="utf-8-sig",
        nrows=100_000,
        low_memory=False,
    )

    # Для сравнения используем нормализованное название.
    name_col = next(
        (col for col in df.columns if col.lower() == "name"),
        None,
    )

    if name_col is None:
        print(f"Нет колонки name: {filename}")
        continue

    df["_normalized_name"] = (
        df[name_col]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    epic_frames[filename] = df

    epic_summary.append({
        "file": filename,
        "rows_loaded": len(df),
        "unique_names": df["_normalized_name"].nunique(),
        "duplicate_names": int(
            df["_normalized_name"].duplicated().sum()
        ),
    })

display(pd.DataFrame(epic_summary))

,file,rows_loaded,unique_names,duplicate_names
0,epic games store-video games.csv,899,899,0
1,epic-games-apps.csv,11,11,0
2,epic-games-demo.csv,25,21,4
3,epic_games_editor.csv,20,18,1


In [15]:
epic_name_sets = {
    filename: set(
        df["_normalized_name"].dropna()
    )
    for filename, df in epic_frames.items()
}

overlap_records = []

for i, (file_a, names_a) in enumerate(epic_name_sets.items()):
    for file_b, names_b in list(epic_name_sets.items())[i + 1:]:
        common = names_a & names_b

        overlap_records.append({
            "file_a": file_a,
            "file_b": file_b,
            "common_names": len(common),
            "share_of_a_pct": round(
                100 * len(common) / len(names_a), 2
            ) if names_a else 0,
            "share_of_b_pct": round(
                100 * len(common) / len(names_b), 2
            ) if names_b else 0,
            "examples": list(sorted(common))[:5],
        })

display(
    pd.DataFrame(overlap_records).sort_values(
        "common_names", ascending=False
    )
)

,file_a,file_b,common_names,share_of_a_pct,share_of_b_pct,examples
1,epic games store-video games.csv,epic-games-demo.csv,3,0.33,14.29,"[bonfire peaks, ghost recon breakpoint, startup panic]"
0,epic games store-video games.csv,epic-games-apps.csv,0,0.00,0.00,[]
2,epic games store-video games.csv,epic_games_editor.csv,0,0.00,0.00,[]
3,epic-games-apps.csv,epic-games-demo.csv,0,0.00,0.00,[]
4,epic-games-apps.csv,epic_games_editor.csv,0,0.00,0.00,[]
5,epic-games-demo.csv,epic_games_editor.csv,0,0.00,0.00,[]


In [16]:
import ijson
from collections import Counter

games_json_path = RAW_DIR / "steam" / "fronkongames" / "games.json"

field_total = Counter()
field_missing = Counter()
game_count = 0

def is_missing_value(value):
    return (
        value is None
        or (isinstance(value, str) and not value.strip())
    )

with games_json_path.open("rb") as file:
    for app_id, game in ijson.kvitems(file, ""):
        game_count += 1

        for field, value in game.items():
            field_total[field] += 1

            if is_missing_value(value):
                field_missing[field] += 1

missing_records = []

for field, total in field_total.items():
    missing = field_missing[field]

    missing_records.append({
        "field": field,
        "records": total,
        "missing": missing,
        "missing_pct": round(
            100 * missing / total, 2
        ) if total else 0,
    })

steam_missing_report = (
    pd.DataFrame(missing_records)
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

print(f"Всего обработано игр: {game_count:,}")
display(steam_missing_report)

steam_missing_report.to_csv(
    REPORTS_DIR / "steam_json_missing_values.csv",
    index=False,
    encoding="utf-8-sig",
)

Всего обработано игр: 144,499


,field,records,missing,missing_pct
0,score_rank,144499,144459,99.97
1,metacritic_url,144499,140217,97.04
2,reviews,144499,131850,91.25
3,notes,144499,116973,80.95
4,website,144499,88860,61.50
5,support_url,144499,82534,57.12
6,support_email,144499,24607,17.03
7,about_the_game,144499,8524,5.90
8,detailed_description,144499,8492,5.88
9,short_description,144499,8333,5.77
